In [13]:
import requests
import json
import pandas as pd



In [8]:
def flatten_chain(node):
    names = [node["species"]["name"]]
    for evo in node["evolves_to"]:
        names.extend(flatten_chain(evo))
    return names

In [9]:
def get_evo_stage(pokemon_name, chain_names):
    index = chain_names.index(pokemon_name)

    if len(chain_names) == 1:
        return "no-evolution"
    elif index == 0:
        return "base"
    elif index == len(chain_names) - 1:
        return "final"
    else:
        return "middle"

In [10]:
def obtain_pokemon_info(num):
    x = requests.get(f'https://pokeapi.co/api/v2/pokemon/{num}')

    base_data = x.json()
    name = base_data["name"]
    species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{name}/")

    species_data = species.json()
    chain_url = species_data["evolution_chain"]["url"]
    chain = requests.get(chain_url).json()
    names = flatten_chain(chain["chain"])

    evo_stage = get_evo_stage(name, names)

    habitat = species_data["habitat"]["name"] if species_data["habitat"] else None
    is_legendary = species_data["is_legendary"] or species_data["is_mythical"]
    generation = species_data["generation"]["name"]
    color = species_data["color"]["name"]
    types = [t["type"]["name"] for t in base_data["types"]]
    return(f"{name}/{is_legendary}/{color}/{generation}/{habitat}/{evo_stage}/{types}/{base_data['base_experience']}/{base_data['height']} ")

In [11]:

x = requests.get(f'https://pokeapi.co/api/v2/pokemon/413')

base_data = x.json()
name = base_data["name"]
species = requests.get(f"https://pokeapi.co/api/v2/pokemon-species/{name}/")
print(species.status_code)

404


In [12]:
with open ("poke_data.csv","a") as f:
    for num in range(1,1026):
        try:
            if requests.get(f'https://pokeapi.co/api/v2/pokemon/{num}').status_code == 200:
                f.write(f"{obtain_pokemon_info(num)}\n")
        except Exception as e:
            print(f"Error processing pokemon {num}: {e}")
            continue

Error processing pokemon 386: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 413: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 487: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 492: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 550: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 555: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 641: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 642: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 645: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 647: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 648: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 678: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 681: Expecting value: line 1 column 1 (char 0)
Error processing pokemon 710: Expecting value: line 1 column 1 (

In [14]:
df = pd.read_csv("poke_data.csv",delimiter="/", header=None, names=["name", "is_legendary", "color", "generation", "habitat", "evo_stage", "types", "base_experience", "height"])

In [15]:
df

,name,is_legendary,color,generation,habitat,evo_stage,types,base_experience,height
0,bulbasaur,False,green,generation-i,grassland,base,"['grass', 'poison']",64,7
1,ivysaur,False,green,generation-i,grassland,middle,"['grass', 'poison']",142,10
2,venusaur,False,green,generation-i,grassland,final,"['grass', 'poison']",236,20
3,charmander,False,red,generation-i,mountain,base,['fire'],62,6
4,charmeleon,False,red,generation-i,mountain,middle,['fire'],142,11
...,...,...,...,...,...,...,...,...,...
986,raging-bolt,False,yellow,generation-ix,NaN,no-evolution,"['electric', 'dragon']",295,52
987,iron-boulder,False,gray,generation-ix,NaN,no-evolution,"['rock', 'psychic']",295,15
988,iron-crown,False,blue,generation-ix,NaN,no-evolution,"['steel', 'psychic']",295,16
989,terapagos,True,blue,generation-ix,NaN,no-evolution,['normal'],90,2
